In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

# 3.0 — DPO (Direct Preference Optimisation) on the Instruction-Tuned Model

## Purpose
This notebook implements the **third and final stage** of the BPMN language model training pipeline. It applies **DPO (Direct Preference Optimisation)** to the instruction-tuned model from Notebook 2.0, aligning its outputs with human-preferred responses.

## What is DPO?
DPO is a preference-based alignment technique that trains a model to prefer *chosen* (high-quality) responses over *rejected* (lower-quality) responses for the same prompt. Unlike RLHF (Reinforcement Learning from Human Feedback), DPO works by directly optimising a language model objective — no separate reward model is required.

The loss function is:

$$\mathcal{L}_{DPO} = -\mathbb{E}\left[\log \sigma\left(\beta \cdot \log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \beta \cdot \log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\right)\right]$$

Where $y_w$ is the chosen response, $y_l$ is the rejected response, $\pi_\theta$ is the trained policy, $\pi_{ref}$ is the reference (instruction-tuned) model, and $\beta$ controls the KL penalty.

## What This Notebook Does
1. **Loads** the instruction-tuned LoRA checkpoint and verifies it runs correctly on the training questions.
2. **Loads** a small curated BPMN DPO dataset with preferred/rejected response pairs.
3. **Merges** the instruction LoRA adapter into the base model weights to create a clean reference policy.
4. **Applies a new DPO LoRA adapter** on top of the merged model.
5. **Runs DPO training** using the `DPOTrainer` from the `trl` library.
6. **Saves** the DPO-aligned adapter and tests the final model output.

## Input
- `./tinyllama-instruction/` — LoRA adapter from Notebook 2.0.
- `data/bpmn_dpo_dataset.jsonl` — BPMN preference pairs (prompt, chosen, rejected).

## Output
- `./tinyllama-dpo-step57/` — The DPO-aligned LoRA adapter.

---

## Step 1 — Import Libraries

The cell above imports core libraries: `transformers`, `peft`, and `datasets`, forming the foundation for model loading and dataset handling in this notebook.

In [12]:
base_model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

## Step 2 — Define the Base Model ID

Specify the Hugging Face Hub identifier for TinyLlama. All LoRA adapters in this pipeline are relative to this same base, ensuring correct weight composition when adapters are loaded and merged.

In [13]:

tokenizer = AutoTokenizer.from_pretrained(base_model)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

## Step 3 — Load the Tokenizer

Load the TinyLlama tokenizer and assign `eos_token` as the `pad_token` (since none is defined by default). This tokenizer is shared across all three training stages and is needed for both quick-verification inference and DPO dataset processing.

In [14]:
import torch, os
from peft import PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"

# Find latest checkpoint dynamically — number changes with each retraining
ckpt_dir = "./tinyllama-instruction"
checkpoints = sorted(
    [d for d in os.listdir(ckpt_dir) if d.startswith("checkpoint-")],
    key=lambda x: int(x.split("-")[1])
)
model_path = os.path.join(ckpt_dir, checkpoints[-1])
print("Loading instruction checkpoint:", model_path)


Loading instruction checkpoint: ./tinyllama-instruction\checkpoint-100


## Step 4 — Locate and Load the Instruction Checkpoint

Dynamically finds the latest checkpoint directory inside `./tinyllama-instruction/` (the folder where Notebook 2.0 saved its adapter). Using dynamic discovery ensures this notebook works correctly even if the checkpoint number changes after retraining.

> **Why load base + PEFT separately?** A LoRA checkpoint contains only the *adapter delta weights*, not the full model. Loading it directly with `AutoModelForCausalLM.from_pretrained()` would give random or missing weights, causing NaN logits and a crash during `torch.multinomial`. The correct approach — load the base model first, then overlay the adapter with `PeftModel.from_pretrained()` — is used here.

In [15]:
# The checkpoint only contains LoRA adapter deltas — NOT full model weights.
# Loading it with AutoModelForCausalLM.from_pretrained() gives random/missing weights
# → logits become NaN/inf → torch.multinomial crashes with RuntimeError.
# Correct approach: load base model first, then overlay the PEFT adapter.
base = AutoModelForCausalLM.from_pretrained(
    base_model, torch_dtype=torch.float32, low_cpu_mem_usage=True
)
instruction_model = PeftModel.from_pretrained(base, model_path)
instruction_model = instruction_model.to(device)
instruction_model.eval()
print("Instruction model loaded correctly.")


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2679.50it/s]


Instruction model loaded correctly.


## Step 5 — Quick Verification: Instruction Model Output

Set a test prompt using the Alpaca-style instruction format and tokenise it, ready for generation. This is used immediately below to sanity-check that the instruction-tuned model generates coherent BPMN responses before DPO training begins.

In [16]:
prompt = "What is BPMN and what is its primary goal?"

In [17]:
prompt = "### Instruction:\nDescribe all five Gateway types in BPMN 2.0 and when to use each.\n### Input:\n\n### Response:\n"
inputs = tokenizer(prompt, return_tensors="pt").to(device)


In [18]:
with torch.no_grad():
    outputs = instruction_model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,        # greedy — safe on a small training run
        repetition_penalty=1.2,
    )


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [19]:
new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
print("\nInstruction model output:\n")
print(tokenizer.decode(new_tokens, skip_special_tokens=True))



Instruction model output:

BPMN 2.0 defines five gateway types, all of which are depicted in Figure 1-3:

1. Exclusive gateways (XOR) – Only one outgoing path is taken based on conditions. At merge, the first arriving token passes through immediately. Icon: X inside circle. Use when exactly one of several paths should execute.

2. Inclusive gateways (OR) – One or more outgoing paths are taken based on conditions. At merge, waits for all active incoming tokens to arrive before continuing. Icon: circle inside circle. Use when any combination of paths may be active.

3. Parallel gates (AND) – All outgoing paths are always taken (fork). At merge, waits for all incoming paths to arrive before proceeding. Icon: + inside circle. Use for unconditional parallel execution.

4. Event-based gates – Routes flow based on


prefrence base tuning or preference based alignment

In [20]:

!pip install -U trl
!pip install -U bitsandbytes

   ---------------------------------------- 0.0/630.8 kB ? eta -:--:--
   ---------------------------------------- 630.8/630.8 kB 11.9 MB/s  0:00:00
   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   ---------------------------------------- 527.0/527.0 kB 16.6 MB/s  0:00:00

  Attempting uninstall: datasets

    Found existing installation: datasets 4.5.0

    Uninstalling datasets-4.5.0:

      Successfully uninstalled datasets-4.5.0

   ---------------------------------------- 0/2 [datasets]
   ---------------------------------------- 0/2 [datasets]
   ---------------------------------------- 0/2 [datasets]
   ---------------------------------------- 0/2 [datasets]
   ---------------------------------------- 0/2 [datasets]
   ---------------------------------------- 0/2 [datasets]
   ---------------------------------------- 0/2 [datasets]
   ---------------------------------------- 0/2 [datasets]
   ---------------------------------------- 0/2 [datasets]
   ---


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import torch, os
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import load_dataset
from trl import DPOTrainer, DPOConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cpu


In [22]:
base_model_id = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# Find the latest instruction checkpoint automatically (save_total_limit=1 keeps only one)
ckpt_dir = "./tinyllama-instruction"
checkpoints = sorted(
    [d for d in os.listdir(ckpt_dir) if d.startswith("checkpoint-")],
    key=lambda x: int(x.split("-")[1])
)
instruction_checkpoint = os.path.join(ckpt_dir, checkpoints[-1])
print("Using instruction checkpoint:", instruction_checkpoint)


Using instruction checkpoint: ./tinyllama-instruction\checkpoint-100


In [23]:
full_dpo = load_dataset("json", data_files="data/bpmn_dpo_dataset.jsonl", split="train")

# Keep only entries relevant to our 2 instruction-training questions:
#   Q1: "What is BPMN and what is its primary goal?"           → index 39 (newly added)
#   Q2: "Describe all five Gateway types in BPMN 2.0..."      → indices 4-7 (Parallel, XOR vs OR, Event-Based, XOR no-default)
dataset = full_dpo.select([39, 4, 5, 6, 7])

print(f"DPO dataset: {len(dataset)} examples")
for ex in dataset:
    print(" -", ex["prompt"])


Generating train split: 40 examples [00:00, 291.29 examples/s]


DPO dataset: 5 examples
 - What is BPMN and what is its primary goal?
 - I need to route a process where ALL outgoing paths must execute simultaneously. Which BPMN gateway should I use?
 - Explain the difference between an Exclusive Gateway and an Inclusive Gateway in BPMN.
 - When should I use an Event-Based Gateway instead of a regular Exclusive Gateway?
 - What happens if no conditions evaluate to true on an Exclusive Gateway that has no default flow?


In [24]:
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


In [25]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [26]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
)


In [27]:
# Load base + merge instruction LoRA into weights
base = AutoModelForCausalLM.from_pretrained(
    base_model_id, torch_dtype=torch.float32, low_cpu_mem_usage=True
)
peft_model = PeftModel.from_pretrained(base, instruction_checkpoint)
merged = peft_model.merge_and_unload()

# Save + reload as plain model — the only way to guarantee no leftover PEFT internals
# (merge_and_unload patches layer classes but doesn't fully strip PEFT state from modules)
MERGED_PATH = "./tinyllama-merged-for-dpo"
merged.save_pretrained(MERGED_PATH)


Writing model shards: 100%|██████████| 1/1 [00:31<00:00, 31.31s/it]


In [28]:
model = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH, torch_dtype=torch.float32, low_cpu_mem_usage=True
)
model = model.to(device)
print("Clean merged model, type:", type(model).__name__)  # Should print LlamaForCausalLM


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2852.20it/s]

Clean merged model, type: LlamaForCausalLM


In [29]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Should show ~4.5M trainable params and NO "second time" warning


trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [30]:

import os
os.environ["WANDB_DISABLED"] = "true"

In [35]:
from trl import DPOTrainer, DPOConfig

In [ ]:
dpo_args = DPOConfig(
    output_dir="./tinyllama-dpo",
    num_train_epochs=20,            # 20 × 4 examples = 80 steps
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=5e-5,             # DPO uses lower LR than SFT to stay close to reference
    beta=0.1,                       # KL penalty — controls how far policy drifts from reference
    max_length=512,
    use_cpu=True,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    save_total_limit=1,
    report_to="none",
    remove_unused_columns=False,
    save_steps=10,
)


In [36]:
trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)


Tokenizing train dataset: 100%|██████████| 5/5 [00:00<00:00, 73.82 examples/s]


In [37]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,0.411420
20,0.011949
30,0.000440
40,0.000160
50,0.000102


KeyboardInterrupt: 

In [38]:
# Save model at step 57
model.save_pretrained("./tinyllama-dpo-step57")
tokenizer.save_pretrained("./tinyllama-dpo-step57")
print("Saved partial DPO model at step 57")

Saved partial DPO model at step 57


In [ ]:
# Test DPO-aligned model on the gateway question from instruction training
model.eval()
prompt = "### Instruction:\nDescribe all five Gateway types in BPMN 2.0 and when to use each.\n### Input:\n\n### Response:\n"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        repetition_penalty=1.2,
    )

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
print("DPO-aligned model output:\n")
print(tokenizer.decode(new_tokens, skip_special_tokens=True))


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


DPO-aligned model output:

BPMN 2.0 defines five gateway types, all of which are depicted in Figure 1-3:

1. Exclusive Gateway (XOR) – Only one outgoing path is taken based on conditions. At merge node, XOR value from incoming tails is passed through to next token. Icon: circle with pentagon inside. Use when exactly one outcome is desired for a single token. Example: Order confirmation after payment approval.

2. Inclusive Gateway (OR) – One or more outgoing paths are taken based on conditions. At merge node, OR value from incoming tails is passed through to next token. Icon: diamond with pentagon inside. Use when any combination of outgoing paths may be active at any time. Example: Order confirmation after payment approval or delivery status change.

3. Parallel Gateway (AND) – All outgoing paths are always taken (fork


: 